# Eligibility sensitivity

**Purpose:** Repeat the foundational procedures on the 285-trial sensitivity population.

**Required inputs:** Prepared Baček HDF5; 04 foundational results ZIP.

**Outputs:** Held-out sensitivity predictions, metrics and selections.

**Usage:** Run after 04. Files are read from the notebook’s current working directory unless specified in its configuration.


In [ ]:
# 1 — Imports and fixed 285-trial sensitivity configuration
from pathlib import Path
from datetime import datetime, timezone
import io
import csv
import hashlib
import json
import os
import platform
import time
import zipfile

import h5py
import numpy as np
import scipy
import sklearn
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

H5 = Path('bacek_clean_common_angles_metabolic_dataset.h5')
REF_ZIP = Path('04_Bacek_Foundational_Predictive_Models_results.zip')
OUT = Path('12_Bacek_285_Trial_Eligibility_Sensitivity_results')
ARCHIVE = Path('12_Bacek_285_Trial_Eligibility_Sensitivity_results.zip')
SOURCE_SHA256 = 'c73ce561e94de52f1dc86128829c43ef536c4cf6b5f4aa9af1ea060409221fa3'
CHANNELS = ('hip_left', 'hip_right', 'knee_left', 'knee_right',
            'ankle_left', 'ankle_right', 'pelvis_sagittal',
            'pelvis_frontal', 'pelvis_transverse')
ALPHAS = (0.01, 0.1, 1.0, 10.0, 100.0, 1000.0)
PCA_COUNTS = (5, 10, 20, 40)
SPEEDS = (1.1, 0.8, 0.4)
CADENCES = (0.90, 0.95, 1.00, 1.10, 1.20)
OUTER_FOLDS = {
    1: ('S08', 'S10', 'S16', 'S21'),
    2: ('S11', 'S13', 'S17', 'S20'),
    3: ('S01', 'S02', 'S05', 'S18'),
    4: ('S03', 'S07', 'S12', 'S14', 'S19'),
    5: ('S04', 'S06', 'S09', 'S15'),
}
EXPECTED_COUNTS = (55, 53, 56, 65, 56)
FOLD_OF = {subject: fold for fold, members in OUTER_FOLDS.items() for subject in members}
PROCEDURES = ('B0', 'B1', 'B2', 'W-ALL9_FIXED', 'D-ALL9_FIXED',
              'P-ALL9_FIXED', 'G_SELECTED')


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as file:
        for block in iter(lambda: file.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def write_csv(path, rows, columns):
    with path.open('w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)


def rmse(actual, predicted):
    error = np.asarray(predicted, dtype=np.float64) - np.asarray(actual, dtype=np.float64)
    return float(np.sqrt(np.mean(error ** 2, dtype=np.float64)))


def performance(actual, predicted):
    actual = np.asarray(actual, dtype=np.float64)
    predicted = np.asarray(predicted, dtype=np.float64)
    error = predicted - actual
    denom = float(np.sum((actual - actual.mean()) ** 2, dtype=np.float64))
    return {'n_trials': len(actual), 'RMSE_W_per_kg': rmse(actual, predicted),
            'MAE_W_per_kg': float(np.mean(np.abs(error), dtype=np.float64)),
            'bias_W_per_kg': float(np.mean(error, dtype=np.float64)),
            'R2': float(1.0 - np.sum(error**2, dtype=np.float64)/denom) if denom > 0 else None}


def check(condition, message):
    if not bool(condition):
        raise AssertionError(message)

print('Python:', platform.python_version(), '| NumPy:', np.__version__,
      '| scikit-learn:', sklearn.__version__)


In [ ]:
# 2 — Verify the 284-trial reference archive (comparison evidence only)
check(REF_ZIP.is_file(), f'Missing Notebook 04 results archive: {REF_ZIP}')
reference_zip_sha = sha256_file(REF_ZIP)
with zipfile.ZipFile(REF_ZIP) as archive:
    required = {'run_summary.json', 'eligible_trial_ledger.csv',
                'held_out_predictions.csv', 'pooled_metrics.csv',
                'selected_models.csv', 'inner_fold_manifest.csv'}
    check(required.issubset(archive.namelist()), 'Notebook 04 reference files missing')
    reference_summary = json.loads(archive.read('run_summary.json'))
    def reference_csv(name):
        return list(csv.DictReader(io.StringIO(archive.read(name).decode('utf-8-sig'))))
    reference_trials = reference_csv('eligible_trial_ledger.csv')
    reference_predictions = reference_csv('held_out_predictions.csv')
    reference_metrics = reference_csv('pooled_metrics.csv')
    reference_selections = reference_csv('selected_models.csv')
    reference_inner = reference_csv('inner_fold_manifest.csv')
check(reference_summary['status'] == 'PASS', 'Notebook 04 reference status is not PASS')
check(reference_summary['source_sha256'] == SOURCE_SHA256, 'Notebook 04 source mismatch')
check(len(reference_trials) == 284 and len({r['trial_uid'] for r in reference_trials}) == 284,
      'Incorrect Notebook 04 population')
check(len(reference_predictions) == 284 * len(PROCEDURES), 'Incorrect Notebook 04 prediction ledger')
check(len(reference_selections) == 30 and len(reference_inner) == 20,
      'Incomplete Notebook 04 development records')
check({r['procedure'] for r in reference_predictions} == set(PROCEDURES),
      'Notebook 04 contains unexpected procedures')
check(all(int(r['outer_fold']) == FOLD_OF[r['subject_id']] for r in reference_trials),
      'Notebook 04 participant fold assignments mismatch')
check({r['procedure'] for r in reference_metrics} == set(PROCEDURES),
      'Notebook 04 pooled metrics incomplete')
check(len({(r['procedure'], r['trial_uid']) for r in reference_predictions}) ==
      len(reference_predictions), 'Notebook 04 duplicate predictions')
for proc in PROCEDURES:
    check({r['trial_uid'] for r in reference_predictions if r['procedure'] == proc} ==
          {r['trial_uid'] for r in reference_trials}, 'Notebook 04 missing reference trials')
ref_by_uid = {r['trial_uid']: r for r in reference_trials}
ref_metrics_by_procedure = {r['procedure']: r for r in reference_metrics}
ref_selected_by_key = {(int(r['outer_fold']), r['procedure']): r for r in reference_selections}
ref_inner_by_key = {(int(r['outer_fold']), int(r['inner_fold'])): r for r in reference_inner}
print('Verified Notebook 04 reference:', len(reference_trials), 'primary trials;',
      len(reference_predictions), 'predictions')


In [ ]:
# 3 — Read verified unrestricted trials and calculate W/D on 285 trials

def features_from_trial(trial):
    means = []
    n_cycles = None
    for channel in CHANNELS:
        if channel not in trial:
            raise ValueError(f'Missing channel {channel}')
        angle = np.asarray(trial[channel][()], dtype=np.float64)
        check(angle.ndim == 2 and angle.shape[1] == 101, 'Invalid angle shape')
        check(np.isfinite(angle).all(), 'Nonfinite gait angles')
        if n_cycles is None:
            n_cycles = angle.shape[0]
        check(angle.shape[0] == n_cycles, 'Inconsistent cycle count')
        means.append(angle.mean(axis=0, dtype=np.float64))
    m = np.stack(means)
    w = m.reshape(909)
    d = []
    for channel in m:
        for q in range(4):
            d.append(float(np.trapezoid(channel[25*q:25*q+26], dx=1.0)/25.0))
        d.append(float(np.std(channel, ddof=0)))
        d.append(float(np.max(channel) - np.min(channel)))
    return w, np.asarray(d, dtype=np.float64), n_cycles


check(H5.is_file(), f'Dataset not found alongside notebook: {H5}')
source_sha = sha256_file(H5)
check(source_sha == SOURCE_SHA256, 'Source SHA-256 mismatch; stop rather than use a different dataset')

rows, w_rows, d_rows = [], [], []
with h5py.File(H5, 'r') as h5:
    subjects = sorted(h5['subjects'])
    check(subjects == sorted(FOLD_OF), 'Participant identities differ from frozen fold map')
    for subject in subjects:
        for trial_id in sorted(h5['subjects'][subject]['trials']):
            if not (trial_id.startswith('T') and trial_id[1:].isdigit() and
                    1 <= int(trial_id[1:]) <= 15):
                continue
            trial = h5['subjects'][subject]['trials'][trial_id]
            uid = f'{subject}_{trial_id}'
            w, d, n = features_from_trial(trial)
            check(n >= 20, f'{uid}: fewer than 20 prepared cycles')
            for attr in ('valid', 'comparison_eligible', 'unrestricted_comparison_eligible'):
                check(attr in trial.attrs and np.asarray(trial.attrs[attr]).reshape(-1)[0] == 1,
                      f'{uid}: invalid {attr}')
            check('metcost_mean' in trial, f'{uid}: missing corrected target')
            target = np.asarray(trial['metcost_mean'][()])
            check(target.size == 1 and np.issubdtype(target.dtype, np.number)
                  and np.isrealobj(target), f'{uid}: invalid target format')
            y_value = float(target.reshape(-1)[0])
            check(np.isfinite(y_value) and y_value > 0, f'{uid}: invalid target value')
            check('original_cycle_number' in trial, f'{uid}: missing cycle provenance')
            indices = np.asarray(trial['original_cycle_number'][()]).reshape(-1)
            check(len(indices) == n and np.all(np.isfinite(indices)) and
                  np.all(indices == np.floor(indices)) and np.all(np.diff(indices) > 0),
                  f'{uid}: invalid original cycle indices')
            if uid == 'S06_T11':
                check(n == 85, 'S06_T11 must contain exactly 85 cycles')
            else:
                check(n >= 100, f'{uid}: unexpected <100-cycle unrestricted trial')
            number = int(trial_id[1:])
            block, cadence_idx = divmod(number - 1, 5)
            rows.append({'trial_uid': uid, 'subject_id': subject,
                         'trial_id': trial_id, 'outer_fold': FOLD_OF[subject],
                         'retained_cycles': n, 'speed_m_s': SPEEDS[block],
                         'relative_cadence': CADENCES[cadence_idx],
                         'condition_index': number - 1, 'speed_index': block,
                         'observed_W_per_kg': y_value})
            w_rows.append(w)
            d_rows.append(d)

check(len(rows) == 285 and len({r['trial_uid'] for r in rows}) == 285,
      'Sensitivity trial ledger must contain 285 unique trial IDs')
check(sum(r['retained_cycles'] for r in rows) == 60842, 'Sensitivity cycle count mismatch')
check([sum(r['outer_fold'] == i for r in rows) for i in OUTER_FOLDS] == list(EXPECTED_COUNTS),
      'Sensitivity outer-fold trial counts mismatch')
check(len({r['subject_id'] for r in rows}) == 21, 'Incorrect participant count')

Xw = np.stack(w_rows)
Xd = np.stack(d_rows)
y = np.asarray([r['observed_W_per_kg'] for r in rows], dtype=np.float64)
groups = np.asarray([r['subject_id'] for r in rows])
uids = np.asarray([r['trial_uid'] for r in rows])
Xb1 = np.eye(3, dtype=np.float64)[[r['speed_index'] for r in rows]]
Xb2 = np.eye(15, dtype=np.float64)[[r['condition_index'] for r in rows]]
check(Xw.shape == (285, 909) and Xd.shape == (285, 54), 'Feature dimensions mismatch')
check(np.isfinite(Xw).all() and np.isfinite(Xd).all(), 'Invalid constructed features')
check(np.all(Xb1.sum(axis=1) == 1) and np.all(Xb2.sum(axis=1) == 1),
      'Condition encoding error')
print('Validated source:', len(rows), 'sensitivity trials /', len(set(groups)), 'participants')
print('Feature matrices:', 'W', Xw.shape, 'D', Xd.shape, 'B1', Xb1.shape, 'B2', Xb2.shape)

# Confirm the exact primary ledger is retained, plus S06_T11 alone.
current_by_uid = {r['trial_uid']: r for r in rows}
check(set(current_by_uid) == set(ref_by_uid) | {'S06_T11'},
      'Sensitivity population differs from 284-trial reference by more than S06_T11')
check(current_by_uid['S06_T11']['retained_cycles'] == 85 and
      current_by_uid['S06_T11']['outer_fold'] == 5,
      'S06_T11 eligibility or outer-fold assignment changed')
for uid, baseline in ref_by_uid.items():
    current = current_by_uid[uid]
    for field in ('subject_id', 'trial_id', 'outer_fold', 'retained_cycles',
                  'speed_m_s', 'relative_cadence', 'condition_index', 'speed_index'):
        old = int(baseline[field]) if field in ('outer_fold', 'retained_cycles',
                   'condition_index', 'speed_index') else (
              float(baseline[field]) if field in ('speed_m_s', 'relative_cadence') else baseline[field])
        check(current[field] == old, f'{uid}: changed reference field {field}')
    check(current['observed_W_per_kg'] == float(baseline['observed_W_per_kg']),
          f'{uid}: stored target differs from Notebook 04 reference')
print('Verified exact 284-trial overlap, plus S06_T11 in outer fold 5')


In [ ]:
# 4 — Unchanged 36-candidate gait selector and reference models

def candidates():
    baselines = [{'representation': name, 'alpha': alpha, 'n_components': None,
                  'n_features': n_features}
                 for name, n_features in [('B1', 3), ('B2', 15)]
                 for alpha in ALPHAS]
    gait = ([{'representation': name, 'alpha': alpha, 'n_components': None,
              'n_features': n_features}
             for name, n_features in [('W-ALL9', 909), ('D-ALL9', 54)]
             for alpha in ALPHAS] +
            [{'representation': 'P-ALL9', 'alpha': alpha, 'n_components': k,
              'n_features': k}
             for k in PCA_COUNTS for alpha in ALPHAS])
    return baselines, gait


def model_input(candidate):
    return {'B1': Xb1, 'B2': Xb2, 'W-ALL9': Xw,
            'D-ALL9': Xd, 'P-ALL9': Xw}[candidate['representation']]


def model_pipeline(candidate):
    rep = candidate['representation']
    if rep in ('B1', 'B2'):
        return Ridge(alpha=candidate['alpha'], fit_intercept=True)
    steps = [('scale', StandardScaler())]
    if rep == 'P-ALL9':
        steps.append(('pca', PCA(n_components=candidate['n_components'],
                                  whiten=False, random_state=2026)))
    steps.append(('ridge', Ridge(alpha=candidate['alpha'], fit_intercept=True)))
    return Pipeline(steps)


def choose(candidate_scores, gait=False):
    if gait:
        # Exact float64 tie: fewer input features, then candidate ID/numeric values.
        return min(candidate_scores, key=lambda r:
                   (r['inner_pooled_RMSE_W_per_kg'], r['n_features'],
                    r['representation'], r['n_components'] or 0, r['alpha']))
    # B1/B2 exact tie: smaller alpha.
    return min(candidate_scores, key=lambda r:
               (r['inner_pooled_RMSE_W_per_kg'], r['alpha']))


base_candidates, gait_candidates = candidates()
check(len(base_candidates) == 12 and len(gait_candidates) == 36,
      'Incorrect candidate catalogue')
print('Candidates:', len(base_candidates), 'condition references +',
      len(gait_candidates), 'gait configurations per outer fold')


## Refit and evaluate on the entire 285-trial population


In [ ]:
# 5 — Rerun B0/B1/B2, fixed W/D/P and G on the 285-trial population
inner_manifest = []
outer_manifest = []
inner_scores = []
candidate_scores = []
selected_models = []
prediction_ledger = []
fit_timing = []
started_at = datetime.now(timezone.utc).isoformat()
start_time = time.perf_counter()

for outer_fold, test_participants in OUTER_FOLDS.items():
    test_idx = np.flatnonzero(np.isin(groups, test_participants))
    dev_idx = np.flatnonzero(~np.isin(groups, test_participants))
    check(len(test_idx) == EXPECTED_COUNTS[outer_fold-1], 'Unexpected outer test count')
    check(set(groups[dev_idx]).isdisjoint(groups[test_idx]), 'Outer participant leakage')

    inner_partitions = []
    covered = set()
    for inner_fold, (training_local, validation_local) in enumerate(
            GroupKFold(n_splits=4).split(Xw[dev_idx], y[dev_idx], groups[dev_idx]), 1):
        training_idx = dev_idx[training_local]
        validation_idx = dev_idx[validation_local]
        train_people = set(groups[training_idx])
        val_people = set(groups[validation_idx])
        check(train_people.isdisjoint(val_people), 'Inner participant leakage')
        check(val_people.isdisjoint(test_participants), 'Outer participant in inner validation')
        check(covered.isdisjoint(validation_idx.tolist()), 'Duplicate inner-validation trial')
        covered.update(validation_idx.tolist())
        inner_partitions.append((training_idx, validation_idx))
        inner_manifest.append({
            'outer_fold': outer_fold, 'inner_fold': inner_fold,
            'train_participants': ';'.join(sorted(train_people)),
            'validation_participants': ';'.join(sorted(val_people)),
            'outer_test_participants': ';'.join(sorted(test_participants)),
            'n_train_trials': len(training_idx), 'n_validation_trials': len(validation_idx),
        })
    check(covered == set(dev_idx.tolist()), 'Incomplete inner-validation coverage')
    outer_manifest.append({
        'outer_fold': outer_fold,
        'development_participants': ';'.join(sorted(set(groups[dev_idx]))),
        'test_participants': ';'.join(sorted(test_participants)),
        'n_development_trials': len(dev_idx), 'n_test_trials': len(test_idx),
    })

    scored = []
    for candidate in base_candidates + gait_candidates:
        X = model_input(candidate)
        pooled_predictions = np.full(len(dev_idx), np.nan, dtype=np.float64)
        # local positions used to recover the original development-trial order
        local_of = {global_index: pos for pos, global_index in enumerate(dev_idx)}
        for inner_fold, (training_idx, validation_idx) in enumerate(inner_partitions, 1):
            model = model_pipeline(candidate)
            t0 = time.perf_counter()
            model.fit(X[training_idx], y[training_idx])
            fit_seconds = time.perf_counter() - t0
            predicted_train = np.asarray(model.predict(X[training_idx]), dtype=np.float64)
            predicted_val = np.asarray(model.predict(X[validation_idx]), dtype=np.float64)
            check(np.isfinite(predicted_train).all() and np.isfinite(predicted_val).all(),
                  'Nonfinite inner model predictions')
            for index, value in zip(validation_idx, predicted_val):
                pooled_predictions[local_of[int(index)]] = value
            inner_scores.append({
                'outer_fold': outer_fold, 'inner_fold': inner_fold,
                **candidate, 'n_train_trials': len(training_idx),
                'n_validation_trials': len(validation_idx),
                'train_RMSE_W_per_kg': rmse(y[training_idx], predicted_train),
                'validation_RMSE_W_per_kg': rmse(y[validation_idx], predicted_val),
                'fit_seconds': fit_seconds,
            })
        check(np.isfinite(pooled_predictions).all(), 'Incomplete pooled inner predictions')
        record = {'outer_fold': outer_fold, **candidate,
                  'inner_pooled_RMSE_W_per_kg': rmse(y[dev_idx], pooled_predictions)}
        candidate_scores.append(record)
        scored.append(record)

    by_rep = {rep: [s for s in scored if s['representation'] == rep]
              for rep in ('B1','B2','W-ALL9','D-ALL9','P-ALL9')}
    choices = {
        'B1': choose(by_rep['B1']),
        'B2': choose(by_rep['B2']),
        'W-ALL9_FIXED': choose(by_rep['W-ALL9'], gait=True),
        'D-ALL9_FIXED': choose(by_rep['D-ALL9'], gait=True),
        'P-ALL9_FIXED': choose(by_rep['P-ALL9'], gait=True),
        'G_SELECTED': choose([s for s in scored if s['representation'] in
                              ('W-ALL9', 'D-ALL9', 'P-ALL9')], gait=True),
    }

    predictions = {'B0': np.full(len(test_idx), float(np.mean(y[dev_idx], dtype=np.float64)))}
    cache = {}  # Refit identical selected configurations once for this outer fold
    for procedure, choice in choices.items():
        identifier = (choice['representation'], choice['alpha'], choice['n_components'])
        if identifier not in cache:
            X = model_input(choice)
            fitted = model_pipeline(choice)
            t0 = time.perf_counter()
            fitted.fit(X[dev_idx], y[dev_idx])
            fit_seconds = time.perf_counter() - t0
            t1 = time.perf_counter()
            saved_pred = np.asarray(fitted.predict(X[test_idx]), dtype=np.float64)
            predict_seconds = time.perf_counter() - t1
            check(np.isfinite(saved_pred).all(), 'Nonfinite outer prediction')
            cache[identifier] = saved_pred
            fit_timing.append({'outer_fold': outer_fold, 'representation': choice['representation'],
                               'alpha': choice['alpha'], 'n_components': choice['n_components'],
                               'fit_seconds': fit_seconds, 'prediction_seconds': predict_seconds,
                               'n_train_trials': len(dev_idx), 'n_predicted_trials': len(test_idx)})
        predictions[procedure] = cache[identifier]
        selected_models.append({'outer_fold': outer_fold, 'procedure': procedure,
                                'representation': choice['representation'],
                                'alpha': choice['alpha'], 'n_components': choice['n_components'],
                                'n_features': choice['n_features'],
                                'inner_pooled_RMSE_W_per_kg': choice['inner_pooled_RMSE_W_per_kg']})

    for procedure in PROCEDURES:
        check(len(predictions[procedure]) == len(test_idx), 'Missing outer predictions')
        for position, index in enumerate(test_idx):
            prediction_ledger.append({
                'procedure': procedure, 'outer_fold': outer_fold,
                'subject_id': rows[index]['subject_id'],
                'trial_id': rows[index]['trial_id'], 'trial_uid': uids[index],
                'observed_W_per_kg': float(y[index]),
                'predicted_W_per_kg': float(predictions[procedure][position]),
            })
    print(f'Outer fold {outer_fold}: {len(test_idx)} test trials; '
          f'G selected {choices["G_SELECTED"]["representation"]} '
          f'({choices["G_SELECTED"]["n_components"] or "full"} features, '
          f'alpha={choices["G_SELECTED"]["alpha"]})')

elapsed = time.perf_counter() - start_time
print(f'Completed {len(PROCEDURES)} procedures on 285 trials each in {elapsed:.1f} s')


In [ ]:
# 6 — Verify 285-trial coverage and compute separate-population metrics
checks = []

def verify(name, passed):
    checks.append({'check': name, 'status': 'PASS' if passed else 'FAIL'})
    check(passed, name)

verify('exact_prediction_row_count', len(prediction_ledger) == 285 * len(PROCEDURES))
verify('all_20_inner_fold_records', len(inner_manifest) == 20)
verify('candidate_score_count', len(candidate_scores) == 5 * 48)
verify('inner_score_count', len(inner_scores) == 5 * 4 * 48)
verify('all_six_selected_procedures_per_outer_fold', len(selected_models) == 5 * 6)
verify('all_predictions_finite', all(np.isfinite(r['predicted_W_per_kg']) for r in prediction_ledger))
verify('unique_prediction_procedure_trial_keys',
       len({(r['procedure'], r['trial_uid']) for r in prediction_ledger}) == len(prediction_ledger))
verify('all_procedures_same_285_trial_keys', all(
       {r['trial_uid'] for r in prediction_ledger if r['procedure'] == procedure} == set(uids)
       for procedure in PROCEDURES))
verify('all_saved_targets_match_authoritative_trial_rows', all(
       r['observed_W_per_kg'] == rows[next(i for i, uid in enumerate(uids) if uid == r['trial_uid'])]['observed_W_per_kg']
       for r in prediction_ledger))
verify('exact_outer_test_fold_assignment', all(
       r['outer_fold'] == FOLD_OF[r['subject_id']] for r in prediction_ledger))

metric_rows, fold_metric_rows = [], []
for procedure in PROCEDURES:
    selected = [r for r in prediction_ledger if r['procedure'] == procedure]
    metrics = performance([r['observed_W_per_kg'] for r in selected],
                          [r['predicted_W_per_kg'] for r in selected])
    metric_rows.append({'procedure': procedure, **metrics})
    for fold in OUTER_FOLDS:
        these = [r for r in selected if r['outer_fold'] == fold]
        fold_metric_rows.append({'outer_fold': fold, 'procedure': procedure,
                                 **performance([r['observed_W_per_kg'] for r in these],
                                               [r['predicted_W_per_kg'] for r in these])})

verify('metrics_are_finite', all(np.isfinite(m['RMSE_W_per_kg']) and
                                 np.isfinite(m['MAE_W_per_kg']) and
                                 np.isfinite(m['bias_W_per_kg']) for m in metric_rows))

by_name = {r['procedure']: r['RMSE_W_per_kg'] for r in metric_rows}
contrasts = {
    'B0_minus_G_RMSE_W_per_kg': by_name['B0'] - by_name['G_SELECTED'],
    'B2_minus_G_RMSE_W_per_kg': by_name['B2'] - by_name['G_SELECTED'],
}
print('\nPooled held-out trial RMSE (W/kg), descriptive results:')
for metric in metric_rows:
    print(f"  {metric['procedure']:15s} {metric['RMSE_W_per_kg']:.6f}")
print('B0 minus G:', f'{contrasts["B0_minus_G_RMSE_W_per_kg"]:.6f}', 'W/kg')

verify('sensitivity_population_differs_only_by_S06_T11',
       set(uids) == set(ref_by_uid) | {'S06_T11'})
verify('S06_T11_only_in_fold_5',
       all(r['outer_fold'] == 5 for r in prediction_ledger if r['trial_uid'] == 'S06_T11'))
verify('all_285_trial_predictions_for_each_procedure',
       all(sum(r['procedure'] == proc for r in prediction_ledger) == 285 for proc in PROCEDURES))

# Deliberately report 284- and 285-population metrics separately; do not pool,
# append to, or claim a paired evaluation of different trial populations.
reference_metric_keys = ('RMSE_W_per_kg', 'MAE_W_per_kg', 'bias_W_per_kg', 'R2')
comparison_rows = []
for metric in metric_rows:
    old = ref_metrics_by_procedure[metric['procedure']]
    comparison_rows.append({
        'procedure': metric['procedure'],
        'primary_284_n_trials': int(old['n_trials']),
        'sensitivity_285_n_trials': metric['n_trials'],
        **{f'primary_284_{k}': float(old[k]) for k in reference_metric_keys},
        **{f'sensitivity_285_{k}': metric[k] for k in reference_metric_keys},
        'descriptive_RMSE_change_285_minus_284_W_per_kg':
            metric['RMSE_W_per_kg'] - float(old['RMSE_W_per_kg'])
    })

selection_comparison = []
for chosen in selected_models:
    previous = ref_selected_by_key[(chosen['outer_fold'], chosen['procedure'])]
    new_config = (chosen['representation'], chosen['alpha'], chosen['n_components'])
    old_config = (previous['representation'], float(previous['alpha']),
                  int(previous['n_components']) if previous['n_components'] else None)
    selection_comparison.append({
        'outer_fold': chosen['outer_fold'], 'procedure': chosen['procedure'],
        'primary_284_representation': old_config[0], 'primary_284_alpha': old_config[1],
        'primary_284_n_components': old_config[2],
        'sensitivity_285_representation': new_config[0], 'sensitivity_285_alpha': new_config[1],
        'sensitivity_285_n_components': new_config[2],
        'selection_changed': (new_config != old_config),
    })

inner_comparison = []
for manifest in inner_manifest:
    previous = ref_inner_by_key[(manifest['outer_fold'], manifest['inner_fold'])]
    inner_comparison.append({
        'outer_fold': manifest['outer_fold'], 'inner_fold': manifest['inner_fold'],
        'primary_284_validation_participants': previous['validation_participants'],
        'sensitivity_285_validation_participants': manifest['validation_participants'],
        'validation_participants_changed': (
            previous['validation_participants'] != manifest['validation_participants']),
        'primary_284_n_validation_trials': int(previous['n_validation_trials']),
        'sensitivity_285_n_validation_trials': manifest['n_validation_trials'],
    })

verify('five_G_selection_comparisons',
       len([r for r in selection_comparison if r['procedure'] == 'G_SELECTED']) == 5)
verify('20_inner_fold_comparisons', len(inner_comparison) == 20)
verify('seven_separate_population_metric_comparisons', len(comparison_rows) == 7)
print('G-selected RMSE, primary 284:',
      f"{float(ref_metrics_by_procedure['G_SELECTED']['RMSE_W_per_kg']):.6f}")
print('G-selected RMSE, sensitivity 285:', f"{by_name['G_SELECTED']:.6f}")
print('G representation selections changed in',
      sum(r['selection_changed'] for r in selection_comparison
          if r['procedure'] == 'G_SELECTED'), 'of five outer folds')
print('Inner validation participant sets changed in',
      sum(r['validation_participants_changed'] for r in inner_comparison),
      'of 20 partition records')


In [ ]:
# 7 — Save full 285-trial evidence and separate-population comparisons
OUT.mkdir(parents=True, exist_ok=True)

# Fingerprint executable notebook code cells (independent of saved cell outputs).
notebook_path = Path('12_Bacek_285_Trial_Eligibility_Sensitivity.ipynb')
notebook_code_sha256 = None
if notebook_path.is_file():
    notebook_json = json.loads(notebook_path.read_text(encoding='utf-8'))
    code_text = ''.join(''.join(cell['source']) for cell in notebook_json['cells']
                        if cell['cell_type'] == 'code')
    notebook_code_sha256 = hashlib.sha256(code_text.encode('utf-8')).hexdigest()

write_csv(OUT / 'eligible_trial_ledger.csv', rows,
          ['trial_uid','subject_id','trial_id','outer_fold','retained_cycles',
           'speed_m_s','relative_cadence','condition_index','speed_index','observed_W_per_kg'])
write_csv(OUT / 'outer_fold_manifest.csv', outer_manifest,
          ['outer_fold','development_participants','test_participants',
           'n_development_trials','n_test_trials'])
write_csv(OUT / 'inner_fold_manifest.csv', inner_manifest,
          ['outer_fold','inner_fold','train_participants','validation_participants',
           'outer_test_participants','n_train_trials','n_validation_trials'])
write_csv(OUT / 'inner_candidate_scores.csv', inner_scores,
          ['outer_fold','inner_fold','representation','alpha','n_components','n_features',
           'n_train_trials','n_validation_trials','train_RMSE_W_per_kg',
           'validation_RMSE_W_per_kg','fit_seconds'])
write_csv(OUT / 'pooled_candidate_scores.csv', candidate_scores,
          ['outer_fold','representation','alpha','n_components','n_features',
           'inner_pooled_RMSE_W_per_kg'])
write_csv(OUT / 'selected_models.csv', selected_models,
          ['outer_fold','procedure','representation','alpha','n_components','n_features',
           'inner_pooled_RMSE_W_per_kg'])
write_csv(OUT / 'held_out_predictions.csv', prediction_ledger,
          ['procedure','outer_fold','subject_id','trial_id','trial_uid',
           'observed_W_per_kg','predicted_W_per_kg'])
write_csv(OUT / 'pooled_metrics.csv', metric_rows,
          ['procedure','n_trials','RMSE_W_per_kg','MAE_W_per_kg','bias_W_per_kg','R2'])
write_csv(OUT / 'outer_fold_metrics.csv', fold_metric_rows,
          ['outer_fold','procedure','n_trials','RMSE_W_per_kg','MAE_W_per_kg','bias_W_per_kg','R2'])
write_csv(OUT / 'outer_model_timings.csv', fit_timing,
          ['outer_fold','representation','alpha','n_components','fit_seconds',
           'prediction_seconds','n_train_trials','n_predicted_trials'])
write_csv(OUT / 'population_metric_comparison.csv', comparison_rows,
          ['procedure','primary_284_n_trials','sensitivity_285_n_trials'] +
          [f'primary_284_{k}' for k in reference_metric_keys] +
          [f'sensitivity_285_{k}' for k in reference_metric_keys] +
          ['descriptive_RMSE_change_285_minus_284_W_per_kg'])
write_csv(OUT / 'selection_comparison.csv', selection_comparison,
          ['outer_fold','procedure','primary_284_representation','primary_284_alpha',
           'primary_284_n_components','sensitivity_285_representation',
           'sensitivity_285_alpha','sensitivity_285_n_components','selection_changed'])
write_csv(OUT / 'inner_fold_comparison.csv', inner_comparison,
          ['outer_fold','inner_fold','primary_284_validation_participants',
           'sensitivity_285_validation_participants','validation_participants_changed',
           'primary_284_n_validation_trials','sensitivity_285_n_validation_trials'])
write_csv(OUT / 'verification_checks.csv', checks, ['check','status'])

summary = {
    'status': 'PASS' if all(r['status'] == 'PASS' for r in checks) else 'REVIEW_REQUIRED',
    'source_hdf5': H5.name, 'source_sha256': source_sha,
    'population': '285 unrestricted trials including S06_T11 with 85 retained cycles',
    'total_sensitivity_trials': 285, 'sensitivity_cycles': 60842, 'participants': 21,
    'prediction_setting': 'new-person, five fixed participant-grouped outer folds',
    'evaluation_status': 'internally held-out; previously explored Baček folds',
    'primary_target': 'stored corrected trial-level metcost_mean, net W/kg',
    'experimental_scope': 'predefined 285-trial sensitivity of foundational 284-trial procedures',
    'procedures': list(PROCEDURES),
    'alpha_grid': list(ALPHAS), 'pca_component_grid': list(PCA_COUNTS),
    'outer_folds': {str(k): list(v) for k, v in OUTER_FOLDS.items()},
    'inner_splitter': 'sklearn.model_selection.GroupKFold(n_splits=4), shuffle=False',
    'pca': {'whiten': False, 'random_state': 2026, 'svd_solver': 'auto'},
    'candidate_tie_rules': {'B1_B2': 'exact pooled RMSE then smaller alpha',
                             'gait': 'exact pooled RMSE, lower dimension, representation ID, components, alpha'},
    'bootstrap_included': False, 'sensitivity_285_included': True,
    'reference_284_archive': REF_ZIP.name,
    'reference_284_archive_sha256': reference_zip_sha,
    'reference_284_notebook_code_sha256': reference_summary.get('notebook_code_sha256'),
    'sensitivity_scope': 'distinct 285-trial evaluation; descriptive comparison with 284-trial primary',
    'sensitivity_reason': 'inclusion of verified S06_T11 (85 cycles); S06 remains in outer fold 5',
    'inner_partitions': 'GroupKFold(n_splits=4) rederived from all 285 rows on each outer development set',
    'changed_G_selected_outer_folds': sum(r['selection_changed'] for r in selection_comparison
                                         if r['procedure'] == 'G_SELECTED'),
    'G_vs_B0_RMSE_improvement_W_per_kg': contrasts['B0_minus_G_RMSE_W_per_kg'],
    'G_vs_B2_RMSE_improvement_W_per_kg': contrasts['B2_minus_G_RMSE_W_per_kg'],
    'started_at_utc': started_at, 'finished_at_utc': datetime.now(timezone.utc).isoformat(),
    'elapsed_seconds': elapsed,
    'software': {'python': platform.python_version(), 'numpy': np.__version__,
                 'scipy': scipy.__version__, 'scikit_learn': sklearn.__version__,
                 'h5py': h5py.__version__, 'os': platform.platform(),
                 'cpu_count': os.cpu_count(),
                 'kernel_python_executable': __import__('sys').executable},
    'notebook_filename': '12_Bacek_285_Trial_Eligibility_Sensitivity.ipynb',
    'notebook_code_sha256': notebook_code_sha256,
    'code_fingerprint_basis': 'concatenated code-cell sources, UTF-8',
    'verification': {r['check']: r['status'] for r in checks},
}
(OUT / 'run_summary.json').write_text(json.dumps(summary, indent=2, allow_nan=False), encoding='utf-8')

with zipfile.ZipFile(ARCHIVE, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for filename in (
            'eligible_trial_ledger.csv', 'outer_fold_manifest.csv',
            'inner_fold_manifest.csv', 'inner_candidate_scores.csv',
            'pooled_candidate_scores.csv', 'selected_models.csv',
            'held_out_predictions.csv', 'pooled_metrics.csv',
            'outer_fold_metrics.csv', 'outer_model_timings.csv',
            'population_metric_comparison.csv', 'selection_comparison.csv',
            'inner_fold_comparison.csv', 'verification_checks.csv', 'run_summary.json'):
        archive.write(OUT / filename, arcname=filename)
print('\nWrote', ARCHIVE, 'with', len(list(OUT.iterdir())), 'evidence files')
print('Recorded verification status:', summary['status'])
